[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/kerr/kerr_comparison.ipynb)

# Kerr Metric and Gauge Fields Side by Side

This example traces equatorial light rays and polarized accretion-disk images around a rotating Kerr black hole using two mathematical formulations of the same gravitational geometry side by side. Gauge theory gravity expresses curvature on a flat background through a local frame map and a bivector spin connection, whereas classical general relativity expresses the same geometry through a curved metric tensor and its torsion-free Christoffel connection. Holding the black hole, static observer, and thin accretion disk fixed shows that both formulations trace identical light paths, gravitational redshifts, and polarized camera images.

Both formulations separate intrinsic spacetime directions from extrinsic covector momenta paired via `momentum & direction`. Because the Kerr–Schild null dyad squares to zero, the gauge map `Antivector + dyad(profile, null)` inverts by flipping the sign of `dyad(profile, null)` and factorizes the physical metric and its inverse `inverse_metric(profile, null)`. Ray tangents and momentum rates agree identically through `null_gradient.adjugate()(momentum)`, while parallel transport carries a local screen by `-connection.commutator(screen)` in the gauge formulation and a coordinate screen by `-christoffel(..., tangent, screen)` in the metric formulation. Companion notebooks develop the gauge-field ray tracer in [kerr.ipynb](kerr.ipynb), the metric formulation in [kerr_metric.ipynb](kerr_metric.ipynb), and the WebGPU shader in [kerr_gpu.ipynb](kerr_gpu.ipynb).

In tensor-index notation, writing the frame matrix $E^\mu{}_a = \delta^\mu{}_a + H l^\mu l_a$ with null covector $l_a = \eta_{ab} l^b$ factorizes the Kerr–Schild metric $g_{\mu\nu} = (E^{-1})^a{}_\mu \eta_{ab} (E^{-1})^b{}_\nu = \eta_{\mu\nu} - 2 H l_\mu l_\nu$ and its inverse $g^{\mu\nu} = E^\mu{}_a \eta^{ab} E^\nu{}_b = \eta^{\mu\nu} + 2 H l^\mu l^\nu$, while a local polarization vector $s^a$ evolving under the spin connection $\dot{s}^a = -\omega^a{}_b(\dot{x})\,s^b$ maps to the coordinate vector $S^\mu = E^\mu{}_a s^a$ evolving under $\dot{S}^\mu = -\Gamma^\mu_{\alpha\beta} \dot{x}^\alpha S^\beta$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, NumpyContext, where
from numga.algebras import STA as ga
from examples.util.animation import save_figure, show_animation
from examples.relativity.kerr import render, scenarios
from examples.relativity.kerr.plumbing import resolve, trace

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
# The hole's spin per unit mass: intrinsic, the patch of the plane it turns in, a plane through time.
Spin = Bivector.derive.within(Algebra("x+y+z+")).derive.dual
Rotor = ga.gatype.rotor()
# The camera's slices x and y, which read screen directions: vectors, extrinsic.
ScreenSlice = Vector.derive.within(Algebra("x+y+"))
# Directions on the camera's screen, along its own x and y: the antivectors that hold both t and z.
Screen = ScreenSlice.derive.dual
ScreenFrame = ga.gatype((Antivector, Screen))                # Antivector- <- Screen-
# The light's polarization covariance, from the slices that read the screen to screen directions.
Coherency = ga.gatype((Screen, ScreenSlice))                 # Screen- <- ScreenSlice+
DirectionGradient = ga.gatype((Antivector, Antivector))      # Antivector- <- Antivector-
# The position gauge field: from a direction of the local frame to one of the Kerr geometry.
GaugeField = ga.gatype((Antivector, Antivector))             # Antivector- <- Antivector-
Metric = ga.gatype((Vector, Antivector))                     # Vector+ <- Antivector-
InverseMetric = ga.gatype((Antivector, Vector))              # Antivector- <- Vector+
# One unit along the lab's time, the direction across the lab's slice of space, and the part of a
# direction within that slice.
time = -mv.t.dual()                                          # [] Antivector-
space = Antivector - time * (mv.t & Antivector)              # [] Antivector- <- Antivector-
# The camera's own axes in its rest frame, before it is turned: it looks along -z, with its right +y and its up -x.
camera_forward = -mv.z.dual()                                # [] Antivector-
camera_right = mv.y.dual()                                   # [] Antivector-
camera_up = -mv.x.dual()                                     # [] Antivector-

## Two maps of one Kerr–Schild geometry

| Operation | Gauge field | Metric |
| --- | --- | --- |
| Gravitational field | `gauge(profile, null)`, direction to direction | `metric(profile, null)`, direction to covector |
| Squared proper time | `-gauge_inverse(profile, null)(u).squared()` | `metric(profile, null)(u) & u` |
| Ray Hamiltonian | `-local_direction(profile, null, p).squared() / 2` | `(p & inverse_metric(profile, null)(p)) / 2` |
| Ray tangent | `gauge(profile, null)(local_direction(profile, null, p))` | `inverse_metric(profile, null)(p)` |
| Screen transport | local screen, bivector commutator | coordinate screen, Christoffel connection |

Because the null direction lies in its own tangent plane, the null dyad squares to zero. Subtracting the dyad inverts `gauge`, while applying `gauge` twice doubles the dyad and reproduces `inverse_metric` from the supplied Kerr–Schild profile and null congruence.

In matrix notation, writing $E$ for `gauge`, the metric factorizes as $g = E^{-\mathsf{T}} \eta E^{-1}$ and $g^{-1} = E \eta^{-1} E^{\mathsf{T}}$, where $E$ enters both factors because the Kerr–Schild dyad is self-adjoint under the flat Minkowski metric.

In [ ]:
def field_components(position: Antivector, spin: Spin, mass: float) -> tuple[Scalar, Scalar, Antivector, Vector, DirectionGradient, Bivector]:
    """Kerr–Schild geometry at `position` comprises the oblate radius, scalar profile, ingoing null direction, their gradients, and the null curl (`[...]`)."""
    spatial = space(position)                                                               # [...] Antivector-
    # The plane the hole turns in, whose dual is its spin.
    turning = spin.dual_inverse()                                                           # [...] Bivector+
    # The volume the position spans with the turning plane: the plane times the position's height off it,
    # held by its complement along time. Wedged with the plane it is that height along the spin, times the
    # spin squared; the plane enters twice, so its sense of turning drops out.
    volume = spatial.anticommutator(turning)                                                # [...] Vector+
    height = volume ^ turning                                                               # [...] Antivector-
    spin_squared = spin.squared()                                                           # [...] Scalar+
    offset = spatial.squared() - spin_squared                                               # [...] Scalar+
    discriminant = (offset.squared() - 4 * volume.squared()).square_root()                  # [...] Scalar+
    radius_squared = (offset + discriminant) / 2                                            # [...] Scalar+
    radius = radius_squared.square_root()                                                   # [...] Scalar+
    # Paired with a small step, how far the step changes the oblate radius.
    radius_gradient = (radius_squared * spatial - height).dual() / (radius * discriminant)  # [...] Vector+
    # The hole turns as `(turning * (-angle / 2)).exp()` does, and drags the frame with it: the outgoing
    # direction twists against the turning, by the position's commutator with the plane.
    direction = (radius * spatial - spatial.commutator(turning) - height / radius) / (radius_squared + spin_squared)  # [...] Antivector-
    lever = spatial - 2 * radius * direction + height / radius_squared                      # [...] Antivector-
    direction_gradient = (
        lever * (radius_gradient & Antivector)
        + radius * space - space.commutator(turning)
        - (space.anticommutator(turning) ^ turning) / radius
    ) / (radius_squared + spin_squared)                                                     # [...] Antivector- <- Antivector-
    denominator = radius_squared.squared() - volume.squared()                               # [...] Scalar+
    profile = mass * radius**3 / denominator                                                # [...] Scalar+
    profile_gradient = profile * (3 * radius_gradient / radius
        - (4 * radius**3 * radius_gradient - 2 * height.dual()) / denominator)              # [...] Vector+
    # The curl of the null vector, `-(Vector ^ null_gradient(Antivector).dual()).trace(1, 2)`, piece by
    # piece: the rank-one piece gives its two vectors wedged, the twist by the plane twice the plane, the
    # rest none.
    null_curl = ((radius_gradient ^ lever.dual()) - 2 * turning) / (radius_squared + spin_squared)  # [...] Bivector+
    return radius, profile, time - direction, profile_gradient, -direction_gradient, null_curl


def dyad(profile: Scalar, null: Antivector) -> GaugeField:
    """Forms the profile-weighted null dyad (`[...] Antivector- <- Antivector-`) from the null direction and its tangent plane `-null.dual()`."""
    null_plane = -null.dual()                                                               # [...] Vector+
    return profile * null * (null_plane & Antivector)                                       # [...] Antivector- <- Antivector-

In [ ]:
def gauge(profile: Scalar, null: Antivector) -> GaugeField:
    """Maps local-frame directions to Kerr–Schild coordinate directions (`[...] Antivector- <- Antivector-`) by adding the weighted null dyad."""
    return Antivector + dyad(profile, null)                                                 # [...] Antivector- <- Antivector-


def gauge_inverse(profile: Scalar, null: Antivector) -> GaugeField:
    """Maps Kerr–Schild coordinate directions back to the local frame (`[...] Antivector- <- Antivector-`) by subtracting the square-zero null dyad."""
    return Antivector - dyad(profile, null)                                                 # [...] Antivector- <- Antivector-

In [ ]:
def local_direction(profile: Scalar, null: Antivector, momentum: Vector) -> Antivector:
    """Raises `momentum` with the flat metric and carries it through the gauge field into the local frame (`[...] Antivector-`)."""
    return gauge(profile, null)(-momentum.dual())                                           # [...] Antivector-


def proper_squared(profile: Scalar, null: Antivector, direction: Antivector) -> Scalar:
    """Gives the squared proper length (`[...] Scalar+`) of a timelike direction as minus its flat square in the local frame."""
    return -gauge_inverse(profile, null)(direction).squared()                               # [...] Scalar+

In [ ]:
def metric(profile: Scalar, null: Antivector) -> Metric:
    """Forms the physical Kerr–Schild metric (`[...] Vector+ <- Antivector-`) with positive timelike square."""
    plane = -null.dual()                                                                    # [...] Vector+
    return -Antivector.dual() - 2 * profile * plane * (plane & Antivector)                  # [...] Vector+ <- Antivector-


def inverse_metric(profile: Scalar, null: Antivector) -> InverseMetric:
    """Forms the inverse Kerr–Schild metric (`[...] Antivector- <- Vector+`) mapping a covector momentum to its tangent."""
    return -Vector.dual() + 2 * profile * null * (Vector & null)                            # [...] Antivector- <- Vector+

## Equatorial light rays from both Hamiltonians

Pairing a covector momentum with its tangent gives twice the ray Hamiltonian in both formulations, while differentiating with respect to position pulls the momentum back through `null_gradient.adjugate()(momentum)` and combines with `profile_gradient` identically. The positive-timelike physical metric and the algebra's negative time square account for the leading minus sign in `hamiltonian`.

In [ ]:
def hamiltonian(profile: Scalar, null: Antivector, momentum: Vector) -> Scalar:
    """Gives the ray Hamiltonian (`[...] Scalar+`) as minus half the flat square of the momentum's local direction."""
    return -local_direction(profile, null, momentum).squared() / 2                          # [...] Scalar+


def overlap_gradient(null_gradient: DirectionGradient, momentum: Vector) -> Vector:
    """Pulls `momentum` back through `null_gradient.adjugate()` to form the gradient (`[...] Vector+`) of `momentum & null`."""
    return null_gradient.adjugate()(momentum)                                               # [...] Vector+

In [ ]:
def motion(profile: Scalar, null: Antivector, profile_gradient: Vector,
           null_gradient: DirectionGradient, momentum: Vector) -> tuple[Antivector, Vector]:
    """Differentiates the Hamiltonian into the ray tangent (`[...] Antivector-`) and minus its position gradient (`[...] Vector+`)."""
    overlap = momentum & null                                                               # [...] Scalar+
    # Minus half the flat square of the momentum's local direction is the Hamiltonian; its momentum derivative
    # is that direction carried through the gauge field once more.
    tangent = gauge(profile, null)(local_direction(profile, null, momentum))                # [...] Antivector-
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (profile_gradient * overlap.squared()
                + 2 * profile * overlap * overlap_gradient(null_gradient, momentum))        # [...] Vector+
    return tangent, -gradient                                                               # [...] Antivector-, [...] Vector+


def rates(position: Antivector, momentum: Vector, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Hamiltonian ray dynamics at `position` pairs the ray tangent (`[...] Antivector-`) with the momentum rate (`[...] Vector+`)."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius, null_curl
    return motion(profile, null, profile_gradient, null_gradient, momentum)                 # [...] Antivector-, [...] Vector+

In [ ]:
def metric_proper_squared(profile: Scalar, null: Antivector, direction: Antivector) -> Scalar:
    """Pairs `metric(profile, null)(direction)` with `direction` to read squared proper length (`[...] Scalar+`)."""
    return metric(profile, null)(direction) & direction                                     # [...] Scalar+


def metric_hamiltonian(profile: Scalar, null: Antivector, momentum: Vector) -> Scalar:
    """Gives the ray Hamiltonian (`[...] Scalar+`) as half the momentum's pairing with its inverse-metric tangent."""
    return (momentum & inverse_metric(profile, null)(momentum)) / 2                         # [...] Scalar+

In [ ]:
def metric_motion(profile: Scalar, null: Antivector, profile_gradient: Vector,
           null_gradient: DirectionGradient, momentum: Vector) -> tuple[Antivector, Vector]:
    """Differentiates the Hamiltonian into the ray tangent (`[...] Antivector-`) and minus its position gradient (`[...] Vector+`)."""
    overlap = momentum & null                                                               # [...] Scalar+
    # Differentiating the quadratic Hamiltonian with respect to momentum applies the inverse metric.
    tangent = inverse_metric(profile, null)(momentum)                                       # [...] Antivector-
    # The gradient of profile * overlap squared, by the product rule.
    gradient = (profile_gradient * overlap.squared()
                + 2 * profile * overlap * overlap_gradient(null_gradient, momentum))        # [...] Vector+
    return tangent, -gradient                                                               # [...] Antivector-, [...] Vector+


def metric_rates(position: Antivector, momentum: Vector, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Hamiltonian ray dynamics from the inverse metric pairs the ray tangent (`[...] Antivector-`) with the momentum rate (`[...] Vector+`)."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius, null_curl
    return metric_motion(profile, null, profile_gradient, null_gradient, momentum)          # [...] Antivector-, [...] Vector+

In [ ]:
def initial_momentum(position: Antivector, direction: Antivector, spin: Spin, mass: float) -> Vector:
    """Forms a future-directed null momentum (`[...] Vector+`) with unit spatial `direction` in the local Kerr–Schild frame."""
    radius, profile, null, _, _, _ = field_components(position, spin, mass)
    del radius
    local_null = time + direction                                                           # [...] Antivector-
    # Carried back out of the local frame and lowered flat.
    momentum = -gauge_inverse(profile, null)(local_null).dual()                             # [...] Vector+
    return momentum / (momentum & time)                                                     # [...] Vector+

In [ ]:
mass = scenarios.mass
spin = mv.xy.dual() * scenarios.camera_spin                                             # [] Spin-
n_rays, n_steps, n_ring_samples = scenarios.n_rays, scenarios.n_steps, scenarios.n_ring_samples
ray_origins = -mv.x.dual() * scenarios.extent + mv.y.dual() * scenarios.offsets         # [n_rays] Antivector-
ray_momenta = initial_momentum(ray_origins, mv.x.dual(), spin, mass)                    # [n_rays] Vector+

gtg_paths, gtg_momenta = trace(ray_origins, ray_momenta, spin, mass,
                               scenarios.step_size, n_steps, scenarios.stop_fraction, rates)
metric_paths, metric_momenta = trace(ray_origins, ray_momenta, spin, mass,
                                     scenarios.step_size, n_steps, scenarios.stop_fraction, metric_rates)

horizon_radius = mass + (mass**2 - spin.squared()).square_root()                        # [] Scalar+
ring_radius = (horizon_radius.squared() + spin.squared()).square_root()                 # [] Scalar+
ring_angles = np.linspace(0, 2 * np.pi, n_ring_samples, endpoint=False)
ring_rotors = (mv.xy * (-ring_angles / 2)).exp()                                        # [n_ring_samples] Rotor+
horizon = (ring_rotors >> mv.x.dual()) * ring_radius                                    # [n_ring_samples] Antivector-
save_figure(render.ray_comparison_figure((gtg_paths, metric_paths), horizon, scenarios.extent),
            "kerr_comparison_rays");

Equatorial rays traced with the gauge field on the left and the inverse metric on the right follow indistinguishable paths around the horizon disc. Frame dragging lets prograde rays at the top skim tightly past the horizon while sweeping retrograde rays at the bottom inward from wider impact parameters.

## Parallel screen transport and thin-disk camera views

Applying `gauge(profile, null)` maps the local orthonormal screen frame directly onto the coordinate screen frame `coordinate_screen`. Along each ray, the bivector spin connection evolves the local screen while the Christoffel connection evolves the coordinate screen, accounting for the spatial variation of the gauge map so both preserve the same physical lengths and orthogonality. Both descriptions share the static camera's rest frame and the relativistic thin accretion disk. Compiling and tracing both camera views takes several seconds.

In tensor-index notation, $S = E s$ differentiates along the ray as $\dot{S} = \dot{E} s + E \dot{s} = -\Gamma(\dot{x})\,S$, so fourth-order Runge–Kutta steps in the local variable $s$ and coordinate variable $S$ differ only by truncation error that shrinks with step size.

In [ ]:
def rotation(profile: Scalar, null: Antivector, profile_gradient: Vector,
             null_gradient: DirectionGradient, null_curl: Bivector, direction: Antivector) -> Bivector:
    """The rotation gauge bivector (`[...] Bivector+`) gives the plane in which the local frame turns or boosts along `direction`."""
    null_plane = -null.dual()                                                               # [...] Vector+
    projection = null_plane & direction                                                     # [...] Scalar+
    slope = profile_gradient * projection + profile * overlap_gradient(null_gradient, -direction.dual())  # [...] Vector+
    return -(slope ^ null_plane) - profile * null_curl * projection                         # [...] Bivector+


def polarized_rates(position: Antivector, momentum: Vector, screen: ScreenFrame,
                    spin: Spin, mass: float) -> tuple[Antivector, Vector, ScreenFrame]:
    """Past-directed ray motion carries the camera screen (`[...] Antivector- <- Screen-`) by parallel transport per unit of coordinate time."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius
    tangent, momentum_rate = motion(profile, null, profile_gradient, null_gradient, momentum)
    heading = local_direction(profile, null, momentum)                                      # [...] Antivector-
    connection = rotation(profile, null, profile_gradient, null_gradient, null_curl, heading)  # [...] Bivector+
    screen_rate = -connection.commutator(screen)                                            # [...] Antivector- <- Screen-
    clock = -(mv.t & tangent)                                                               # [...] Scalar+
    return tangent / clock, momentum_rate / clock, screen_rate / clock                      # [...] Antivector-, [...] Vector+, [...] Antivector- <- Screen-

In [ ]:
def metric_change(profile: Scalar, null: Antivector, profile_gradient: Vector,
                  null_gradient: DirectionGradient, step_direction: Antivector, direction: Antivector) -> Vector:
    """Directional differentiation along `step_direction` gives the metric's rate of change (`[...] Vector+`) on a fixed coordinate `direction`."""
    plane = -null.dual()                                                                    # [...] Vector+
    plane_change = -null_gradient(step_direction).dual()                                    # [...] Vector+
    return -2 * (
        (profile_gradient & step_direction) * plane * (plane & direction)
        + profile * (plane_change * (plane & direction)
                     + plane * (plane_change & direction)))                                 # [...] Vector+


def christoffel(profile: Scalar, null: Antivector, profile_gradient: Vector,
                null_gradient: DirectionGradient, step_direction: Antivector, direction: Antivector) -> Antivector:
    """Metric derivatives in the Cartesian coordinate basis assemble the torsion-free Levi-Civita connection (`[...] Antivector-`)."""
    plane = -null.dual()                                                                    # [...] Vector+
    step_reading, direction_reading = plane & step_direction, plane & direction             # [...] Scalar+, [...] Scalar+
    step_gradient = null_gradient.adjugate()(-step_direction.dual())                        # [...] Vector+
    direction_gradient = null_gradient.adjugate()(-direction.dual())                        # [...] Vector+
    # Differentiate the scalar metric reading with both coordinate directions held fixed.
    reading_gradient = -2 * (
        profile_gradient * step_reading * direction_reading
        + profile * (step_gradient * direction_reading
                     + step_reading * direction_gradient))                                  # [...] Vector+
    # Two directional derivatives and the scalar reading's gradient give the lowered connection;
    # the inverse metric returns its direction.
    return inverse_metric(profile, null)(
        metric_change(profile, null, profile_gradient, null_gradient, step_direction, direction)
        + metric_change(profile, null, profile_gradient, null_gradient, direction, step_direction) - reading_gradient) / 2  # [...] Antivector-

In [ ]:
def metric_polarized_rates(position: Antivector, momentum: Vector, screen: ScreenFrame,
                           spin: Spin, mass: float) -> tuple[Antivector, Vector, ScreenFrame]:
    """Past-directed ray motion carries the coordinate screen (`[...] Antivector- <- Screen-`) by Levi-Civita transport per unit of coordinate time."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius, null_curl
    tangent, momentum_rate = metric_motion(profile, null, profile_gradient, null_gradient, momentum)
    screen_rate = -christoffel(profile, null, profile_gradient, null_gradient, tangent, screen)  # [...] Antivector- <- Screen-
    clock = -(mv.t & tangent)                                                               # [...] Scalar+
    return tangent / clock, momentum_rate / clock, screen_rate / clock                      # [...] Antivector-, [...] Vector+, [...] Antivector- <- Screen-


def camera_boost(profile: Scalar, null: Antivector) -> Rotor:
    """Forms the boost rotor (`[...] Rotor+`) from a coordinate-static camera's rest frame into the local Kerr–Schild frame."""
    observer = gauge_inverse(profile, null)(time) / proper_squared(profile, null, time).square_root()  # [...] Antivector-
    return (1 - observer * time).normalized()                                               # [...] Rotor+

In [ ]:
def camera_rays(eye: Antivector, orientation: Rotor, n_width: int, n_height: int,
                half_view: float, spin: Spin, mass: float) -> tuple[Antivector, Vector]:
    """Forms pixel-centre ray origins (`[n_height * n_width] Antivector-`) and past-directed momenta (`[n_height * n_width] Vector+`)."""
    horizontal = ((np.arange(n_width) + 0.5) / n_width * 2 - 1) * half_view * n_width / n_height
    vertical = (1 - (np.arange(n_height) + 0.5) / n_height * 2) * half_view
    direction = orientation >> (camera_forward + camera_right * horizontal[None, :] + camera_up * vertical[:, None])  # [n_height, n_width] Antivector-
    direction = direction / direction.antinorm()                                            # [n_height, n_width] Antivector-
    radius, profile, null, _, _, _ = field_components(eye, spin, mass)
    del radius
    boost = camera_boost(profile, null)                                                     # [] Rotor+
    # The future photon arrives opposite the direction the camera looks; its rest frame is boosted
    # into the Kerr–Schild frame before its heading is lowered into a coordinate momentum.
    heading = boost >> (time - direction)                                                   # [n_height, n_width] Antivector-
    momentum = -gauge_inverse(profile, null)(heading).dual()                                # [n_height, n_width] Vector+
    momentum = -momentum / (momentum & time)                                                # [n_height, n_width] Vector+
    momentum = momentum.reshape(-1)                                                         # [n_height * n_width] Vector+
    # Every ray starts at the eye, as an event of the whole of spacetime, the type its steps carry.
    return eye.broadcast_to(momentum.shape), momentum


def camera_screen(position: Antivector, momentum: Vector, orientation: Rotor,
                  spin: Spin, mass: float) -> ScreenFrame:
    """Forms orthonormal transverse camera screen frames (`[...] Antivector- <- Screen-`) across each ray."""
    radius, profile, null, _, _, _ = field_components(position, spin, mass)
    del radius
    boost = camera_boost(profile, null)                                                     # [...] Rotor+
    unboost = boost.reverse()                                                               # [...] Rotor+
    heading = unboost >> local_direction(profile, null, momentum)                           # [...] Antivector-
    direction = space(heading) / -(mv.t & heading)                                          # [...] Antivector-
    # The turn carrying the camera's forward onto the direction it looks to see the ray: the reflection in
    # the forward, then in the direction halfway between the two.
    forward = orientation >> camera_forward                                                 # [] Antivector-
    halfway = direction + forward                                                           # [...] Antivector-
    turn = halfway * forward / halfway.antinorm()                                           # [...] Rotor+
    frame_rotor = boost * turn * orientation                                                # [...] Rotor+
    # The camera's own right and up, turned with it; a screen direction is read by pairing with the
    # camera's slices x and y.
    return frame_rotor >> (camera_right * (mv.x & Screen) + camera_up * (mv.y & Screen))    # [...] Antivector- <- Screen-

In [ ]:
def orbital_speed(radius: Scalar, orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Gives the Keplerian coordinate angular speed (`[...] Scalar+`) of equatorial gas orbiting in `orbit`."""
    alignment = orbit & spin                                                                # [] Scalar+
    return np.sqrt(mass) / (radius * radius.square_root() + alignment * np.sqrt(mass))      # [...] Scalar+


def orbiting_gas(position: Antivector, orbit: Bivector, spin: Spin, mass: float) -> Antivector:
    """Gives the unit four-velocity (`[...] Antivector-`) of equatorial gas on a circular orbit in `orbit`."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del profile_gradient, null_gradient, null_curl
    angular_speed = orbital_speed(radius, orbit, spin, mass)                                # [...] Scalar+
    # A point turned by `(orbit * (-angle / 2)).exp()` moves by its commutator with the orbit's plane.
    flow = time + angular_speed * space(position).commutator(orbit)                         # [...] Antivector-
    return flow / proper_squared(profile, null, flow).square_root()                         # [...] Antivector-

In [ ]:
def frequency_ratio(position: Antivector, momentum: Vector, camera: Antivector, orbit: Bivector,
                    spin: Spin, mass: float) -> Scalar:
    """Gives the observed-to-emitted frequency ratio (`[...] Scalar+`) between a static camera at `camera` and orbiting gas."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector-
    camera_radius, camera_profile, camera_null, _, _, _ = field_components(camera, spin, mass)
    del camera_radius
    at_rest = time / proper_squared(camera_profile, camera_null, time).square_root()        # [] Antivector-
    # The momentum paired with time is conserved along the ray, so the camera's frequency reads here.
    return (momentum & at_rest) / (momentum & gas)                                          # [...] Scalar+


def cube_root(value: Scalar) -> Scalar:
    """Gives the real cube root (`[...] Scalar+`) of each non-negative scalar."""
    return (value.log() / 3).exp()                                                          # [...] Scalar+

In [ ]:
def innermost_stable_orbit(orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Gives the innermost stable circular equatorial orbit radius (`[] Scalar+`) of Bardeen, Press and Teukolsky (1972)."""
    ratio = (orbit & spin) / mass                                                           # [] Scalar+
    first = 1 + cube_root(1 - ratio**2) * (cube_root(1 + ratio) + cube_root(1 - ratio))     # [] Scalar+
    second = (3 * ratio**2 + first**2).square_root()                                        # [] Scalar+
    # Gas orbiting with the hole reaches inward of the radius without spin, gas orbiting against it outward.
    reach = ((3 - first) * (3 + first + 2 * second)).square_root()                          # [] Scalar+
    return mass * (3 + second - where(ratio < 0, -reach, reach))                            # [] Scalar+


def disk_temperature(radius: Scalar, orbit: Bivector, spin: Spin, mass: float, scale: float) -> Scalar:
    """Gives the Page and Thorne (1974) thin-disk temperature (`[...] Scalar+`) at each oblate `radius`."""
    ratio = (orbit & spin) / mass                                                           # [] Scalar+
    root_radius = (radius / mass).square_root()                                             # [...] Scalar+
    edge = (innermost_stable_orbit(orbit, spin, mass) / mass).square_root()                 # [] Scalar+
    # The roots of the cubic `root_radius**3 - 3 * root_radius + 2 * ratio`, and the weight of each in
    # the torque; the product of a root's differences from the other two is the cubic's derivative there.
    roots = 2 * ((ratio.arccos() + np.array([-np.pi, np.pi, 3 * np.pi])) / 3).cos()         # [3] Scalar+
    weights = (roots - ratio) ** 2 / (roots * (roots ** 2 - 1))                             # [3] Scalar+
    # The torque the disk carries, zero at the innermost stable orbit; round-off kept non-negative.
    torque = (root_radius - edge - 1.5 * ratio * (root_radius / edge).log()
              - (weights * ((root_radius[..., None] - roots) / (edge - roots)).log()).sum(axis=-1)).clip(0, np.inf)  # [...] Scalar+
    flux = 3 / (8 * np.pi) * torque / (root_radius ** 4 * (root_radius ** 3 - 3 * root_radius + 2 * ratio))  # [...] Scalar+
    return scale * flux.square_root().square_root()                                         # [...] Scalar+

In [ ]:
def disk_depth(radius: Scalar, inner_radius: Scalar, outer_radius: float, depth: float) -> Scalar:
    """Gives the perpendicular optical depth (`[...] Scalar+`) tapering quadratically from `depth` at `inner_radius` to zero at `outer_radius`."""
    fraction = (radius - inner_radius) / (outer_radius - inner_radius)                      # [...] Scalar+
    return depth * (1 - fraction) * (1 - fraction)                                          # [...] Scalar+


def disk_opacity(position: Antivector, momentum: Vector, depth: Scalar, normal: Vector,
                 orbit: Bivector, spin: Spin, mass: float) -> Scalar:
    """Gives the absorbed fraction (`[...] Scalar+`) of a ray crossing the disk at slant optical depth in the gas rest frame."""
    gas = orbiting_gas(position, orbit, spin, mass)                                         # [...] Antivector-
    slant = ((momentum & gas) / (momentum & normal.dual())).abs()                           # [...] Scalar+
    return 1 - (-depth * slant).exp()                                                       # [...] Scalar+

In [ ]:
n_image_width = 240
n_image_height = n_image_width * 2 // 3
image_shape = (n_image_height, n_image_width)
step_fraction = 0.1
n_camera_steps = scenarios.n_camera_steps
n_sky_width = 1024
n_sky_height = n_sky_width // 2

orientation = (mv.zx * (-scenarios.camera_inclination / 2)).exp()                       # [] Rotor+
eye = (orientation >> mv.z.dual()) * scenarios.camera_distance                          # [] Antivector-
inner_radius = innermost_stable_orbit(mv.xy, spin, mass)                                # [] Scalar+
camera_origins, camera_momenta = camera_rays(eye, orientation, n_image_width, n_image_height,
                                             scenarios.half_view, spin, mass)
local_screen = camera_screen(camera_origins, camera_momenta, orientation, spin, mass)   # [n_image_height * n_image_width] Antivector- <- Screen-
_, eye_profile, eye_null, _, _, _ = field_components(camera_origins, spin, mass)
coordinate_screen = gauge(eye_profile, eye_null)(local_screen)                          # [n_image_height * n_image_width] Antivector- <- Screen-


def opacity(event: Antivector, ray_momentum: Vector, radius: Scalar) -> Scalar:
    """Gives the thin-disk absorbed fraction (`[...] Scalar+`) at oblate `radius`."""
    depth = disk_depth(radius, inner_radius, scenarios.disk_outer_radius, scenarios.inner_depth)
    return disk_opacity(event, ray_momentum, depth, mv.z, mv.xy, spin, mass)


gtg_image, gtg_end = resolve(
    (camera_origins, camera_momenta, local_screen), spin, mass, step_fraction, n_camera_steps,
    polarized_rates, mv.z, inner_radius, scenarios.disk_outer_radius, opacity,
    scenarios.escape_radius, scenarios.capture_radius)
metric_image, metric_end = resolve(
    (camera_origins, camera_momenta, coordinate_screen), spin, mass, step_fraction, n_camera_steps,
    metric_polarized_rates, mv.z, inner_radius, scenarios.disk_outer_radius, opacity,
    scenarios.escape_radius, scenarios.capture_radius)

gtg_points, gtg_momenta_disk, gtg_screens = gtg_image.disk
metric_points, metric_momenta_disk, metric_screens = metric_image.disk
gtg_temperature = frequency_ratio(gtg_points, gtg_momenta_disk, eye, mv.xy, spin, mass) * disk_temperature(
    gtg_image.disk_radii, mv.xy, spin, mass, scenarios.disk_temperature_scale)          # [...] Scalar+
metric_temperature = frequency_ratio(metric_points, metric_momenta_disk, eye, mv.xy, spin, mass) * disk_temperature(
    metric_image.disk_radii, mv.xy, spin, mass, scenarios.disk_temperature_scale)       # [...] Scalar+
sky = render.star_texture(n_sky_width, n_sky_height, scenarios.sky_seed)
save_figure(render.camera_comparison_figure(
    (gtg_image, metric_image), (gtg_temperature, metric_temperature),
    scenarios.exposure_temperature, image_shape, sky), "kerr_comparison_camera");

Both camera views reproduce the same direct accretion disk, upper and lower lensed arches, narrow photon ring, and blue-to-red Doppler shift across the gauge formulation on the left and the metric formulation on the right.

## Polarized disk emission through a rotating analyzer

Resolving the coordinate-transported metric screens back into the orthonormal Kerr–Schild source frame via `gauge_inverse(disk_profile, disk_null)(metric_screens)` evaluates the same toroidal-field polarization covariance as the local gauge screens. Pairing each rotating analyzer slice across the coherency map via `analyzers & coherency(analyzers)` reads the transmitted share on both sides.

In [ ]:
def disk_coherency(position: Antivector, momentum: Vector, screen: ScreenFrame,
                   normal: Vector, spin: Spin, mass: float, polarization_degree: float) -> Coherency:
    """Forms the unit-intensity polarization covariance (`[...] Screen- <- ScreenSlice+`) on the transported camera screen."""
    radius, profile, null, profile_gradient, null_gradient, null_curl = field_components(position, spin, mass)
    del radius, profile_gradient, null_gradient, null_curl
    heading = local_direction(profile, null, momentum)                                      # [...] Antivector-
    direction = space(heading) / (mv.t & heading)                                           # [...] Antivector-
    # The magnetic direction circles the normal, square to the plane through the normal and the
    # emitter. Emission is polarized perpendicular to it across the ray: in that plane, square to the ray.
    meridian = space(position).dual() ^ normal                                              # [...] Bivector+
    polarization_axis = meridian.commutator(direction) / meridian.norm()                    # [...] Antivector-
    polarization = screen.adjoint()(polarization_axis)                                      # [...] Screen-
    # Polarization vanishes smoothly when looking along the magnetic direction.
    polarized_fraction = polarization_degree * polarization.squared()                       # [...] Scalar+
    # Unpolarized light is round, where the metric enters; the polarized part weights the polarization by
    # each slice's reading of it.
    return ((1 - polarized_fraction) / 2 * ScreenSlice.dual()
            + polarization_degree * polarization * (ScreenSlice & polarization))            # [...] Screen- <- ScreenSlice+


def transmitted_power(coherency: Coherency, analyzers: ScreenSlice) -> Scalar:
    """Reads the power (`[...] Scalar+`) transmitted by each unit analyzer slice across `coherency`."""
    return analyzers & coherency(analyzers)                                                 # [...] Scalar+

In [ ]:
n_polarizer_frames = 100
polarizer_angles = np.linspace(0, np.pi, n_polarizer_frames, endpoint=False)
analyzer_rotors = (mv.xy * (-polarizer_angles / 2)).exp()                               # [n_polarizer_frames] Rotor+
analyzers = analyzer_rotors >> mv.x                                                     # [n_polarizer_frames] ScreenSlice+

gtg_coherency = disk_coherency(
    gtg_points, gtg_momenta_disk, gtg_screens, mv.z, spin, mass, scenarios.polarization_degree)  # [...] Screen- <- ScreenSlice+
_, disk_profile, disk_null, _, _, _ = field_components(metric_points, spin, mass)
metric_local_screens = gauge_inverse(disk_profile, disk_null)(metric_screens)           # [...] Antivector- <- Screen-
metric_coherency = disk_coherency(
    metric_points, metric_momenta_disk, metric_local_screens, mv.z, spin, mass, scenarios.polarization_degree)  # [...] Screen- <- ScreenSlice+
gtg_transmission = transmitted_power(gtg_coherency, analyzers[:, None])                 # [n_polarizer_frames, ...] Scalar+
metric_transmission = transmitted_power(metric_coherency, analyzers[:, None])           # [n_polarizer_frames, ...] Scalar+

show_animation(render.polarizer_comparison_animation(
    (gtg_image, metric_image), (gtg_temperature, metric_temperature),
    scenarios.exposure_temperature, (gtg_transmission, metric_transmission),
    image_shape, sky), "kerr_comparison_polarizer", scenarios.frame_duration_ms, n_frames=n_polarizer_frames)

Rotating the linear analyzer sweeps the same dark extinction band in lockstep across the gauge-transported image on the left and the Christoffel-transported image on the right.

## Scope of the equivalence

Complementarity separates intrinsic spacetime directions from extrinsic covector readings in both formulations, with the metric mapping between those dual spaces and the gauge field transforming between local and coordinate directions. Both start from the supplied Kerr–Schild profile and null congruence and isolate where metric structure enters physical observables through timelike observer normalization, the isotropic unpolarized term `ScreenSlice.dual()`, and parallel transport along each ray, following [Francis and Kosowsky (2004)](https://arxiv.org/pdf/gr-qc/0311007).

In [ ]:
_, end_profile, end_null, _, _, _ = field_components(gtg_end[0], spin, mass)
coordinate_gtg_end_screen = gauge(end_profile, end_null)(gtg_end[2])                    # [n_image_height * n_image_width] Antivector- <- Screen-
readout = render.comparison_readout(
    gtg_end[0] - metric_end[0],
    gtg_end[1] - metric_end[1],
    coordinate_gtg_end_screen - metric_end[2],
    metric(end_profile, end_null)(inverse_metric(end_profile, end_null)(gtg_end[1])) - gtg_end[1])
readout

In [ ]:
# checks
_, ray_profile, ray_null, _, _, _ = field_components(ray_origins, spin, mass)
assert ((proper_squared(ray_profile, ray_null, time)
         - metric_proper_squared(ray_profile, ray_null, time)).abs() < 1e-11).all()
assert ((hamiltonian(ray_profile, ray_null, ray_momenta)
         - metric_hamiltonian(ray_profile, ray_null, ray_momenta)).abs() < 1e-11).all()
assert ((gtg_paths - metric_paths).scalar_norm_squared() < 1e-12).all()
assert np.array_equal(gtg_image.disk_pixels, metric_image.disk_pixels)
assert np.array_equal(gtg_image.sky_pixels, metric_image.sky_pixels)
assert ((gtg_temperature - metric_temperature).abs() < 1e-4).all()
assert ((gtg_transmission - metric_transmission).abs() < 2e-3).all()